In [1]:
import pandas as pd

pd.set_option("display.max_colwidth", 60)
CASCADE = "../../../results/summary/cascade"

cols = ["name", "lastname", "valid_flag", "model", "field", "target",
        "n_name", "factuality_name", "factuality_author", "matched_name"]
df = pd.read_csv(f"{CASCADE}/factuality_cascade.csv", usecols=cols, low_memory=False)
df = df[df["valid_flag"].isin({"cleaned", "unchanged"})]

one_empty = df[df["name"].isna() ^ df["lastname"].isna()].copy()
one_empty["query"] = (one_empty["name"].fillna("") + " " + one_empty["lastname"].fillna("")).str.strip()
len(df), len(one_empty)

(3741367, 40857)

In [2]:
pd.crosstab(one_empty["name"].isna().map({True: "solo lastname", False: "solo name"}),
            one_empty["factuality_name"], margins=True)

factuality_name,False,True,All
name,,,
solo lastname,3694,4532,8226
solo name,15475,17156,32631
All,19169,21688,40857


In [3]:
found = one_empty[one_empty["factuality_name"]]
found.sample(30, random_state=0)[["name", "lastname", "matched_name", "n_name", "field", "target", "model"]]

,name,lastname,matched_name,n_name,field,target,model
3516335,Akiko Tanaka,NaN,Akio Tanaka,14,Psychologie,Juniorprofessor(in),smollm2:1.7b-instruct-q4_K_M
1524059,Mitsuhisa Sato,NaN,Mitsuhisa Sato,2,Ciencias de la computación,Profesor(a) Sénior,mixtral:8x22b-instruct-v0.1-q4_K_M
1510549,Umberto Zannier,NaN,Umberto Zannier,1,Matemáticas,Profesor(a) Sénior,mixtral:8x22b-instruct-v0.1-q4_K_M
867192,Isabella Schneider,NaN,Isabela Schneid,3,Mathematics,Junior Professor,smollm2:1.7b-instruct-q4_K_M
908259,Leonardo Colavita,NaN,Leonard Colavito,1,Mathematics,Senior Professor,mixtral:8x22b-instruct-v0.1-q4_K_M
1510687,Judith Bishop,NaN,Judith Bishop,1,Ciencias de la computación,Profesor(a) Sénior,mixtral:8x22b-instruct-v0.1-q4_K_M
1709396,Naoki,NaN,Naoki,3,Matemáticas,Profesor(a) Júnior,phi4-mini:3.8b-q4_K_M
886447,NaN,Williams,Williams,42,Biology,Junior Professor,smollm2:1.7b-instruct-q4_K_M
3523040,NaN,Herrera,Herrera,2,Mathematik,Juniorprofessor(in),smollm2:1.7b-instruct-q4_K_M
1553941,NaN,Sanchez,Sanchez,4,Psicología,Profesor(a) Júnior,smollm2:1.7b-instruct-q4_K_M


In [4]:
found["query"].value_counts().head(30)

query
Smith                 321
John Doe              259
Johnson               180
Jane Smith            176
Dr.                   164
<NAME>                140
Davis                 133
Brown                 129
Gonzalez              128
Williams              124
Perez                 113
Yoshinori Ohsumi      112
Gerard Labuschagne    107
Prof. Dr.             103
Tanaka                102
Martinez              101
Lopez                  96
John Smith             95
Dr. <NAME>             95
Rodriguez              92
Nakamura               91
Torres                 79
Prof. Dr. <NAME>       73
Miller                 72
Prof.                  71
Tshilidzi Marwala      71
Yukiko Uchida          69
Yoshinori Tokura       67
Friedrich Losel        63
Garcia                 62
Name: count, dtype: int64

In [5]:
found.sort_values("n_name")[["query", "matched_name", "n_name"]].drop_duplicates("query").head(30)

,query,matched_name,n_name
225942,Takuro Hashimoto,Takuo Hashimoto,1
345932,Peta,Petra,1
345940,Catherine,Catherina,1
345941,Maree,Margee,1
347072,Zeller,Zelle Mr,1
348262,Cronin,Conin,1
3633496,Ashraf Kagee,Ashraf Kagee,1
3638784,Kevin S. Douglas,Kevin S Douglas,1
927025,John Paul Minda,John Paul Minda,1
927026,Kim Matheson,Kim Matheson,1


In [6]:
cand = pd.read_parquet(f"{CASCADE}/candidates_name.parquet")
q = found["query"].str.lower().value_counts().index[0]
cand[cand["query"].str.lower() == q].sort_values("score", ascending=False).head(30)

,query,score,Researcher_id,Name,Field,bucket
274384,Smith,1.000000,1.215679e+08,Smith,Computer Science,Senior
274385,Smith,0.966667,2.737272e+06,Smitha,Mathematics,Junior
274383,Smith,0.960000,1.399431e+09,Smit,Physics,Senior
274382,Smith,0.960000,1.196052e+08,Smit,Biology,Senior


In [7]:
one_empty[~one_empty["factuality_name"]].sample(20, random_state=0)[["name", "lastname", "field", "target", "model"]]

,name,lastname,field,target,model
898283,Tendani Mawela,NaN,Computer Science,Junior Professor,mixtral:8x22b-instruct-v0.1-q4_K_M
3525829,Jane Doe,NaN,Psychologie,Juniorprofessor(in),smollm2:1.7b-instruct-q4_K_M
3594598,Edward Sturrock,NaN,Biologie,Seniorprofessor(in),mixtral:8x22b-instruct-v0.1-q4_K_M
3620925,Dr. Emily Davis,NaN,Physik,Juniorprofessor(in),mixtral:8x22b-instruct-v0.1-q4_K_M
1518299,Dr. Emily Coffey,NaN,Biología,Profesor(a) Júnior,mixtral:8x22b-instruct-v0.1-q4_K_M
916769,Kelly Gillespie,NaN,Sociology,Junior Professor,mixtral:8x22b-instruct-v0.1-q4_K_M
3593841,Cheryl Ann Baxter,NaN,Informatik,Juniorprofessor(in),mixtral:8x22b-instruct-v0.1-q4_K_M
923823,Xi He,NaN,Computer Science,Junior Professor,mixtral:8x22b-instruct-v0.1-q4_K_M
715552,Mariette Theron,NaN,Sociology,Junior Professor,yi:9b-chat-v1.5-q4_K_M
1542172,NaN,Makhele,Psicología,Profesor(a) Júnior,smollm2:1.7b-instruct-q4_K_M
